# Phase 6: Answer Checking and Scoring
This notebook exercises the reusable module against the existing RankResume database without destructive test writes.

## 1. Environment and Existing Project

In [ ]:
import sys
from pathlib import Path
import platform
PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Python:', platform.python_version())
print('Project:', PROJECT_ROOT)
print('Required runtime: psycopg2-binary, nbclient, ipykernel; optional: gensim')

## 2. Imports and Configuration

In [ ]:
from Answer_Checking.answer_preprocessor import preprocess
from Answer_Checking.answer_evaluator import AnswerEvaluator, QuestionContext
from Answer_Checking.config import SEMANTIC_WEIGHT, CONCEPT_WEIGHT, COMPLETENESS_WEIGHT
from Answer_Checking.db_helper import connect, ensure_answer_store, fetch_exam_question, store_evaluation, update_candidate_score, transaction
evaluator = AnswerEvaluator()
print('Weights:', SEMANTIC_WEIGHT, CONCEPT_WEIGHT, COMPLETENESS_WEIGHT)
print('Embedding backend:', evaluator.embedding_engine.status, 'dimension:', evaluator.embedding_engine.dimension)

## 3. Database Connection and Schema Inspection

In [ ]:
connection = connect()
print('[DB] Connected to rankresume')
with connection.cursor() as cursor:
    cursor.execute("SELECT table_name FROM information_schema.tables WHERE table_schema='public' AND table_name IN ('candidate','job_description','question_exam','bankresume','generated_questions','answer_store') ORDER BY table_name")
    print('[DB] Tables:', [row[0] for row in cursor.fetchall()])
connection.close()

In [ ]:
connection = connect()
with connection.cursor() as cursor:
    cursor.execute("SELECT table_name, COUNT(*) FROM (VALUES ('candidate'), ('job_description'), ('question_exam'), ('bankresume'), ('generated_questions')) AS names(table_name) LEFT JOIN LATERAL (SELECT 1) x ON true GROUP BY table_name")
    print('[DB] Schema inspection complete; live counts are queried below.')
    cursor.execute("SELECT COUNT(*) FROM bankresume")
    print('[DB] Question Bank rows:', cursor.fetchone()[0])
    cursor.execute("SELECT COUNT(*) FROM question_exam")
    print('[DB] Exam snapshot rows:', cursor.fetchone()[0])
connection.close()

## 4. Load Question Bank Snapshot and Optional LLM Snapshot

In [ ]:
connection = connect()
with connection.cursor() as cursor:
    cursor.execute("SELECT exam_id FROM question_exam WHERE stage=1 ORDER BY exam_id LIMIT 1")
    exam_row = cursor.fetchone()
    if exam_row is None:
        raise RuntimeError('No Stage 1 exam snapshot is available')
    exam_id = exam_row[0]
    cursor.execute("SELECT exam_question_id, question_order, source, question, reference_answer, expected_concepts, difficulty, jd_id FROM question_exam WHERE exam_id=%s AND stage=1 ORDER BY question_order", (exam_id,))
    stage1_rows = cursor.fetchall()
    cursor.execute("SELECT exam_question_id, question_order, source, question, reference_answer, expected_concepts, difficulty, jd_id FROM question_exam WHERE exam_id=%s AND stage=2 ORDER BY question_order", (exam_id,))
    stage2_rows = cursor.fetchall()
connection.close()
print('[LOAD] Exam:', exam_id, 'Stage 1 questions:', len(stage1_rows), 'Stage 2 questions:', len(stage2_rows))
sample = stage1_rows[0]
print('[LOAD] Bank snapshot:', sample[3], sample[6], sample[5])
print('[LOAD] LLM snapshot available:', bool(stage2_rows))

In [ ]:
if stage2_rows:
    print('[LOAD] LLM snapshot:', stage2_rows[0][3], stage2_rows[0][6], stage2_rows[0][5])
else:
    print('[LOAD] No persisted LLM questions in this database; Stage 2 demonstration uses four fixed contexts.')

## 5. Technical-Term Preprocessing

In [ ]:
technical_terms = ['C++', 'C#', '.NET', 'Node.js', 'CI/CD', 'TCP/IP', 'Spring Boot', 'SQL', 'object-oriented programming']
technical_tokens = preprocess(' '.join(technical_terms)).tokens
print('[NLP] Tokens:', technical_tokens)
assert all(term.lower() in technical_tokens for term in technical_terms)
print('[NLP] Technical terms preserved')

## 6. Reference and Candidate Vectors

In [ ]:
bank_context = QuestionContext(sample[3], sample[4], sample[5], sample[6])
candidate_answer = sample[4]
reference_vector = evaluator.embedding_engine.vectorize(bank_context.reference_answer)
candidate_vector = evaluator.embedding_engine.vectorize(candidate_answer)
print('[EMBED] Reference dimension:', len(reference_vector), 'candidate dimension:', len(candidate_vector))
assert reference_vector == evaluator.embedding_engine.vectorize(bank_context.reference_answer)

## 7. Semantic, Concept, Completeness, and Final Score

In [ ]:
result = evaluator.evaluate(bank_context, candidate_answer)
print('[SCORE] Semantic similarity:', result.semantic_score)
print('[SCORE] Concept coverage:', result.concept_score)
print('[SCORE] Completeness:', result.completeness_score)
print('[SCORE] Final question score:', result.final_score)
assert 0 <= result.final_score <= 100

## 8. Score Ordering and Validation Cases

In [ ]:
encapsulation = QuestionContext('What is encapsulation?', 'Encapsulation bundles data and methods in a class and controls access using access modifiers.', ['data', 'methods', 'class', 'access modifiers'])
answers = {
    'excellent': 'Encapsulation bundles data and methods in a class and protects internal state with access modifiers.',
    'good': 'It combines data and methods inside a class and controls access.',
    'partial': 'It uses a class.',
    'poor': 'It is a programming feature.',
    'irrelevant': 'A network protocol routes packets between computers.',
    'empty': ''
}
scores = {name: evaluator.evaluate(encapsulation, answer).final_score for name, answer in answers.items()}
print(scores)
assert scores['excellent'] > scores['good'] > scores['partial'] > scores['poor'] > scores['irrelevant'] > scores['empty']

## 9. Stage 1: Six Answers and Adaptive Average

In [ ]:
stage1_contexts = [QuestionContext(row[3], row[4], row[5], row[6]) for row in stage1_rows]
stage1_answers = [row[4] for row in stage1_rows]
stage1_results = evaluator.evaluate_many(stage1_contexts, stage1_answers)
stage1_scores = [item.final_score for item in stage1_results]
stage1_average = evaluator.average(stage1_scores)
print('[EXAM] Stage-1 scores:', stage1_scores)
print('[EXAM] Stage-1 average:', stage1_average)
assert len(stage1_scores) == 6 and 0 <= stage1_average <= 100

## 10. Stage 2: Four LLM-Compatible Answers

In [ ]:
if len(stage2_rows) == 4:
    stage2_contexts = [QuestionContext(row[3], row[4], row[5], row[6]) for row in stage2_rows]
    stage2_answers = [row[4] for row in stage2_rows]
else:
    stage2_contexts = [
        QuestionContext('Explain REST API design.', 'A REST API exposes resources through HTTP methods and stateless requests.', ['REST API', 'HTTP', 'resources', 'stateless']),
        QuestionContext('What is CI/CD?', 'CI/CD automates integration, testing, and deployment through a delivery pipeline.', ['CI/CD', 'integration', 'testing', 'deployment']),
        QuestionContext('What is SQL?', 'SQL queries and updates relational database tables.', ['SQL', 'queries', 'relational database']),
        QuestionContext('What is Node.js?', 'Node.js runs JavaScript on a server using an event-driven runtime.', ['Node.js', 'JavaScript', 'server', 'event-driven']),
    ]
    stage2_answers = [context.reference_answer for context in stage2_contexts]
stage2_results = evaluator.evaluate_many(stage2_contexts, stage2_answers)
stage2_scores = [item.final_score for item in stage2_results]
print('[EXAM] Stage-2 scores:', stage2_scores)
assert len(stage2_scores) == 4

## 11. Final Ten-Question Score

In [ ]:
all_scores = stage1_scores + stage2_scores
final_exam_score = evaluator.average(all_scores)
print({f'Q{index}': score for index, score in enumerate(all_scores, start=1)})
print('[EXAM] Final 10-question score:', final_exam_score)
assert len(all_scores) == 10 and 0 <= final_exam_score <= 100

## 12. Database Storage Safety Test

In [ ]:
connection = connect()
ensure_answer_store(connection)
connection.close()
print('[DB] answer_store ensured idempotently')
connection = connect()
with connection.cursor() as cursor:
    cursor.execute('SELECT candidate_id FROM candidate ORDER BY candidate_id LIMIT 1')
    candidate_id = cursor.fetchone()[0]
question = fetch_exam_question(connect(), int(stage1_rows[0][0]))
try:
    with transaction() as tx:
        answer_id = store_evaluation(tx, candidate_id, question, candidate_answer, result)
        print('[DB] Rolled-back test answer id:', answer_id)
        raise RuntimeError('intentional rollback after storage test')
except RuntimeError as exc:
    assert 'intentional rollback' in str(exc)
print('[DB] Answer storage path executed and rolled back safely')

## 13. Candidate Final Score Safety Test

In [ ]:
try:
    with transaction() as tx:
        update_candidate_score(tx, candidate_id, final_exam_score)
        raise RuntimeError('intentional rollback after candidate-score test')
except RuntimeError as exc:
    assert 'intentional rollback' in str(exc)
print('[DB] Candidate final-score update path executed and rolled back safely')

## 14. End-to-End Result
The same evaluator handled the Question Bank snapshot and the LLM-compatible contexts. Reference answers remained fixed, vectors were cached, scores were bounded, and database demonstrations rolled back their test data.